In [0]:

from pathlib import Path
import sys

# Environment configuration
def get_notebook_path():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
    except Exception as e:
        raise RuntimeError(
            "Não foi possível identificar o caminho do notebook."
        ) from e


NOTEBOOK_PATH = get_notebook_path()
NOTEBOOK_PATH_LOWER = NOTEBOOK_PATH.lower()

if "/prod/" in NOTEBOOK_PATH_LOWER:
    ENV = "prod"
elif "/dev/" in NOTEBOOK_PATH_LOWER:
    ENV = "dev"
else:
    raise ValueError(
        f"Ambiente não identificado pelo caminho: {NOTEBOOK_PATH}"
    )

print(f"Environment selected: {ENV.upper()}")


# Python package configuration (DEV)
if ENV == "dev":
    workspace_path = (
        NOTEBOOK_PATH
        if NOTEBOOK_PATH.startswith("/Workspace/")
        else f"/Workspace{NOTEBOOK_PATH}"
    )

    notebook_dir = Path(workspace_path).parent

    PROJECT_ROOT = next(
        (
            parent
            for parent in notebook_dir.parents
            if (parent / "pyproject.toml").is_file()
        ),
        None
    )

    if PROJECT_ROOT is not None:
        SRC_PATH = PROJECT_ROOT / "src"

        if SRC_PATH.is_dir() and str(SRC_PATH) not in sys.path:
            sys.path.insert(0, str(SRC_PATH))

        print(f"Python source path: {SRC_PATH}")
    else:
        print("Project source directory not found; using installed packages.")


# Catalog configuration
CATALOGS = {
    "bronze": f"{ENV}_bronze",
    "silver": f"{ENV}_silver",
    "gold": f"{ENV}_gold"
}


# Schema configuration
SCHEMAS = {
    "bronze": "warframe_market",
    "staging": "staging",
    "intermediate": "intermediate",
    "gold": "marts"
}


# Table configuration
def get_table(layer: str, table: str) -> str:
    catalog_key = {
        "bronze": "bronze",
        "staging": "silver",
        "intermediate": "silver",
        "gold": "gold"
    }.get(layer)

    if catalog_key is None:
        raise ValueError(f"Invalid data layer: {layer}")

    return f"{CATALOGS[catalog_key]}.{SCHEMAS[layer]}.{table}"


def get_gold_table(schema: str, object_name: str) -> str:
    valid_schemas = {"intermediate", "marts", "reporting"}
    if schema not in valid_schemas:
        raise ValueError(f"Invalid Gold schema: {schema}")
    return f"{CATALOGS['gold']}.{schema}.{object_name}"


# Raw volume configuration
RAW_VOLUME = f"{CATALOGS['bronze']}.warframe_market.raw"

RAW_PATH = (
    f"/Volumes/{CATALOGS['bronze']}/"
    "warframe_market/raw/warframe_market/"
)


# Configuration summary
print(f"Environment: {ENV.upper()}")
print(f"Catalogs: {list(CATALOGS.values())}")
print(f"Raw volume: {RAW_VOLUME}")
print(f"Raw path: {RAW_PATH}")